# DCR Pareto RL — Training Analysis\n\nThis notebook loads all `train_trace_*.csv` files from a training run and provides:\n- **Summary tables** with exact numbers per weight pair\n- **Interactive plots** (Plotly) for reward, trace length, cost and duration\n- **Pareto front** visualisation and data table\n\nSet `LOGS_DIR` to wherever your CSVs are."

In [ ]:
import re, csv
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── CONFIG ─────────────────────────────────────────────────────────────────
LOGS_DIR  = Path("logs")          # change to your local path if needed
SMOOTH_W  = 100                   # smoothing window in episodes
# ───────────────────────────────────────────────────────────────────────────

COLORS = {
    "α=0.0 β=0.0": "#2196F3",
    "α=1.0 β=0.0": "#4CAF50",
    "α=0.0 β=1.0": "#F44336",
    "α=0.5 β=0.5": "#FF9800",
    "α=2.0 β=0.5": "#9C27B0",
    "α=0.5 β=2.0": "#00BCD4",
}

def parse_weights(exp_id):
    m = re.search(r'_a([\dp]+)_b([\dp]+)', exp_id)
    if m:
        return float(m.group(1).replace("p",".")), float(m.group(2).replace("p","."))
    return None, None

def wlabel(a, b):
    return f"α={a:.1f} β={b:.1f}" if a is not None else "unknown"

def smooth(v, w):
    if w <= 1 or len(v) < w: return np.array(v)
    return np.convolve(v, np.ones(w)/w, mode='valid')

def load_data(logs_dir):
    rows = []
    for p in sorted(logs_dir.glob("train_trace_*.csv")):
        exp_id = p.stem.replace("train_trace_exp_","")
        a, b = parse_weights(exp_id)
        label = wlabel(a, b)
        with open(p, newline="") as f:
            for row in csv.DictReader(f):
                if str(row.get("done","")).lower() not in ("true","1"):
                    continue
                try:
                    rows.append({
                        "label":      label, "alpha": a, "beta": b,
                        "episode":    int(row.get("episode", 0)),
                        "timestep":   int(row.get("global_timestep", 0)),
                        "reward":     float(row.get("ep_rew_sum", 0)),
                        "steps":      int(row.get("episode_steps", 0)),
                        "cost":       float(row["episode_cost"])     if row.get("episode_cost")     not in ("","None",None) else None,
                        "duration":   float(row["episode_duration"]) if row.get("episode_duration") not in ("","None",None) else None,
                        "accepting":  str(row.get("accepting","")).lower() in ("true","1"),
                        "illegal":    int(row.get("illegal_traces_count", 0)),
                    })
                except (ValueError, KeyError):
                    pass
    return pd.DataFrame(rows)

df = load_data(LOGS_DIR)
print(f"Loaded {len(df):,} episodes from {df['label'].nunique()} weight pairs")
df.head(3)

## 1 · Summary table — exact numbers per weight pair"

In [ ]:
acc = df[df["accepting"]]

summary = df.groupby("label").agg(
    total_episodes   = ("episode",  "count"),
    accepting_episodes = ("accepting", "sum"),
    final_reward_mean  = ("reward",  lambda x: x.iloc[-200:].mean()),
    final_reward_last  = ("reward",  lambda x: x.iloc[-1]),
    trace_len_mean     = ("steps",   "mean"),
    trace_len_min      = ("steps",   "min"),
    trace_len_final    = ("steps",   lambda x: x.iloc[-200:].mean()),
).round(2)

summary["accept_rate_%"] = (summary["accepting_episodes"] / summary["total_episodes"] * 100).round(1)

acc_stats = acc.groupby("label").agg(
    cost_mean     = ("cost",     "mean"),
    cost_min      = ("cost",     "min"),
    cost_final    = ("cost",     lambda x: x.dropna().iloc[-200:].mean() if len(x.dropna()) > 0 else None),
    duration_mean = ("duration", "mean"),
    duration_min  = ("duration", "min"),
    duration_final= ("duration", lambda x: x.dropna().iloc[-200:].mean() if len(x.dropna()) > 0 else None),
).round(2)

full = summary.join(acc_stats)
full.style.background_gradient(cmap="RdYlGn", subset=["accept_rate_%", "final_reward_mean"])\
          .background_gradient(cmap="RdYlGn_r", subset=["cost_final","duration_final"])\
          .format(precision=2)

## 2 · Reward convergence"

In [ ]:
def interactive_plot(df, metric, title, ylabel, accepting_only=False, smooth_w=SMOOTH_W):
    fig = go.Figure()
    sub = df[df["accepting"]] if accepting_only else df
    for label, grp in sub.groupby("label"):
        vals = grp[metric].dropna().values
        if len(vals) == 0: continue
        color = COLORS.get(label, "#888")
        x = list(range(len(vals)))
        # raw (faint)
        fig.add_trace(go.Scatter(x=x, y=vals, mode="lines",
            line=dict(color=color, width=1), opacity=0.15,
            showlegend=False, hoverinfo="skip"))
        # smoothed
        sv = smooth(vals, smooth_w)
        sx = list(range(len(smooth(vals,1)) - len(sv), len(smooth(vals,1))))
        hover = [f"<b>{label}</b><br>Episode: {sx[i]:,}<br>{metric}: {sv[i]:.2f}" for i in range(len(sv))]
        fig.add_trace(go.Scatter(x=list(range(len(sv))), y=sv, mode="lines",
            name=label, line=dict(color=color, width=2.5),
            hovertemplate="%{customdata}<extra></extra>", customdata=hover))
    fig.update_layout(title=title, xaxis_title="Episode", yaxis_title=ylabel,
        legend_title="α=cost, β=duration", plot_bgcolor="white",
        xaxis=dict(showgrid=True, gridcolor="#eee"),
        yaxis=dict(showgrid=True, gridcolor="#eee"), height=480)
    fig.show()

interactive_plot(df, "reward", "Reward convergence per weight pair", "Episode reward")

## 3 · Trace length, Cost and Duration"

In [ ]:
interactive_plot(df,  "steps",    "Trace length per episode",                    "Steps")
interactive_plot(df,  "cost",     "Episode cost — accepting traces only",         "Total cost",     accepting_only=True)
interactive_plot(df,  "duration", "Episode duration — accepting traces only",     "Total duration", accepting_only=True)

## 4 · Pareto front"

In [ ]:
def dominates(a, b):
    return a[0] <= b[0] and a[1] <= b[1] and (a[0] < b[0] or a[1] < b[1])

def pareto_front(points):
    seen, unique = {}, []
    for p in points:
        k = (p["cost"], p["duration"])
        if k not in seen: seen[k] = p; unique.append(p)
    front = [c for c in unique if not any(
        dominates((p["cost"],p["duration"]),(c["cost"],c["duration"])) for p in unique if p is not c)]
    return sorted(front, key=lambda p: p["cost"])

acc_cd = df[df["accepting"] & df["cost"].notna() & df["duration"].notna()]
points = acc_cd[["label","cost","duration","steps","episode","timestep"]].to_dict("records")
front  = pareto_front(points)

# --- Scatter ---
fig = go.Figure()
for label, grp in acc_cd.groupby("label"):
    color = COLORS.get(label, "#888")
    fig.add_trace(go.Scatter(
        x=grp["cost"], y=grp["duration"], mode="markers",
        name=label, marker=dict(color=color, size=5, opacity=0.3),
        hovertemplate=f"<b>{label}</b><br>Cost: %{{x:.1f}}<br>Duration: %{{y:.1f}}<extra></extra>"))

if front:
    fx = [p["cost"] for p in front]
    fy = [p["duration"] for p in front]
    fhover = [f"<b>Pareto</b><br>Cost: {p['cost']:.1f}<br>Duration: {p['duration']:.1f}<br>From: {p['label']}" for p in front]
    fig.add_trace(go.Scatter(x=fx, y=fy, mode="markers+lines",
        name="Pareto front", marker=dict(color="black", size=12, symbol="star"),
        line=dict(color="black", width=2, dash="dot"),
        hovertemplate="%{customdata}<extra></extra>", customdata=fhover))

fig.update_layout(title="Pareto Front — Cost vs Duration (accepting traces)",
    xaxis_title="Total Cost", yaxis_title="Total Duration",
    legend_title="α=cost, β=duration", plot_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="#eee"),
    yaxis=dict(showgrid=True, gridcolor="#eee"), height=520)
fig.show()

# --- Table ---
print(f"\nPareto front: {len(front)} unique points\n")
pd.DataFrame(front)[["label","cost","duration","steps","episode"]]\
  .rename(columns={"label":"weight","steps":"trace_len","episode":"ep_num"})\
  .sort_values("cost")